In [ ]:
import openai
import json

# 1. 定义工具（天气查询函数，实际中可能调用第三方API）
def get_weather(city, date=None):
    """模拟查询天气的工具函数"""
    if not date:
        date = "2025-09-02"  # 默认当天
    # 模拟API返回结果
    return {
        "city": city,
        "date": date,
        "temperature": "25℃",
        "condition": "晴"
    }

# 2. 工具描述（提供给模型）
functions = [
    {
        "name": "get_weather",
        "description": "查询指定城市的天气情况",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "城市名称"},
                "date": {"type": "string", "description": "日期，格式YYYY-MM-DD"}
            },
            "required": ["city"]
        }
    }
]

# 3. 与模型交互的主逻辑
def chat_with_function(user_query):
    # 第一步：向模型提问，要求判断是否需要调用工具
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": user_query}],
        functions=functions,  # 告诉模型可用的工具
        function_call="auto"  # 让模型自动决定是否调用工具
    )
    
    # 解析模型响应
    message = response["choices"][0]["message"]
    
    # 如果模型决定调用工具
    if "function_call" in message:
        function_name = message["function_call"]["name"]
        function_args = json.loads(message["function_call"]["arguments"])
        
        # 调用对应的工具函数
        if function_name == "get_weather":
            result = get_weather(**function_args)
            
            # 第二步：将工具结果回传给模型，让其整理回答
            final_response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "user", "content": user_query},
                    message,  # 模型之前的调用指令
                    {"role": "function", "name": function_name, "content": json.dumps(result)}
                ]
            )
            return final_response["choices"][0]["message"]["content"]
    
    # 如果不需要调用工具，直接返回回答
    return message["content"]

# 测试：查询北京明天的天气
print(chat_with_function("北京2025-09-03的天气怎么样？"))
# 输出：北京2025-09-03的天气为晴，气温25℃。
